### **Importing libraries**

In [45]:
import numpy as np
import pandas as pd
import GEOparse

from scipy.stats import ttest_ind
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import SelectKBest, f_classif

### **Load dataset from GEO.soft file**
Replace the path with your actual file location

In [3]:
import logging

logging.getLogger("GEOparse").setLevel(logging.ERROR)

In [4]:
gse = GEOparse.get_GEO(
    filepath="../data/GSE18123_family.soft"
)

c:\Users\Asrefanavary\AppData\Local\Programs\Python\Python313\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")



### **Inspecting the GEO Sample**

The first GEO sample (GSM) is extracted from the downloaded dataset, and its expression matrix is examined. The dataset contains 54,613 probes, with each row representing a probe identifier (ID_REF) and its corresponding normalized expression value (VALUE).

In [5]:
gsm = list(gse.gsms.values())[0]

print(gsm.table.shape)
gsm.table.head()

(54613, 2)


,ID_REF,VALUE
0,1007_s_at,26.65640
1,1053_at,39.00726
2,117_at,296.01724
3,121_at,209.03575
4,1255_g_at,63.82730


### **Extracting Sample Metadata**

The metadata of each GEO sample (GSM) are extracted to build a structured sample information table. For every sample, the diagnosis, gender, age at blood collection, and microarray platform are retrieved from the available metadata and stored in a pandas DataFrame.

The resulting table contains **285 samples** and **5 metadata fields** (`Sample`, `Diagnosis`, `Gender`, `Age`, and `Platform`), providing the clinical and demographic information required for subsequent data preprocessing and analysis.


In [6]:
samples_info = []

for gsm_id, gsm in gse.gsms.items():

    diagnosis = None
    gender = None
    age = None

    for item in gsm.metadata.get("characteristics_ch1", []):

        if item.startswith("diagnosis:"):
            diagnosis = item.replace("diagnosis:", "").strip()

        elif item.startswith("gender:"):
            gender = item.replace("gender:", "").strip()

        elif item.startswith("age at blood drawing"):
            age = int(item.split(":")[-1].strip())

    samples_info.append({
        "Sample": gsm_id,
        "Diagnosis": diagnosis,
        "Gender": gender,
        "Age": age,
        "Platform": gsm.metadata["platform_id"][0]
    })

df_samples = pd.DataFrame(samples_info)

In [7]:
df_samples

,Sample,Diagnosis,Gender,Age,Platform
0,GSM650510,PDD-NOS,male,118,GPL570
1,GSM650512,PDD-NOS,male,79,GPL570
2,GSM650513,AUTISM,male,169,GPL570
3,GSM650514,AUTISM,male,92,GPL570
4,GSM650515,AUTISM,male,78,GPL570
...,...,...,...,...,...
280,GSM650959,CONTROL,male,28,GPL6244
281,GSM650968,CONTROL,male,51,GPL6244
282,GSM650969,CONTROL,male,37,GPL6244
283,GSM650970,CONTROL,male,49,GPL6244


### **Selecting the GPL6244 Platform**

The dataset contains samples generated using two different microarray platforms: **GPL570 (99 samples)** and **GPL6244 (186 samples)**. 
Since **GPL6244** includes a substantially larger number of samples, it was selected for the subsequent analyses. Using the platform with the larger sample size provides greater statistical power and ensures a more comprehensive analysis while avoiding potential technical variability introduced by combining data from different platforms.

In [8]:
df_samples["Platform"].unique()

<StringArray>
['GPL570', 'GPL6244']
Length: 2, dtype: str

In [9]:
df_samples["Platform"].value_counts()

Platform
GPL6244    186
GPL570      99
Name: count, dtype: int64

In [10]:
df_gpl6244 = df_samples[df_samples["Platform"] == "GPL6244"].copy()

df_gpl6244.shape

(186, 5)

In [11]:
df_gpl6244.head(10)

,Sample,Diagnosis,Gender,Age,Platform
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244
100,GSM650656,AUTISM,male,168,GPL6244
101,GSM650657,CONTROL,male,228,GPL6244
102,GSM650658,AUTISM,male,60,GPL6244
103,GSM650659,AUTISM,male,36,GPL6244
104,GSM650660,PDD-NOS,male,72,GPL6244
105,GSM650661,PDD-NOS,male,108,GPL6244
106,GSM650662,ASPERGER'S DISORDER,male,120,GPL6244
107,GSM650663,PDD-NOS,male,48,GPL6244
108,GSM650664,PDD-NOS,male,108,GPL6244


### **Diagnostic Grouping Strategy**

The original dataset contained four diagnostic categories: **CONTROL**, **PDD-NOS**, **AUTISM**, and **ASPERGER'S DISORDER**. According to the **DSM-IV** criteria, Autism, Asperger's Disorder, and PDD-NOS were classified as pervasive developmental disorders and represent related neurodevelopmental conditions within the autism spectrum.

In the updated **DSM-5** framework, these previously separate diagnoses were integrated into a single clinical category known as **Autism Spectrum Disorder (ASD)**.

The original dataset contained four diagnostic categories:

| Diagnosis | Number of Samples |
|------------|------------------|
| CONTROL | 82 |
| PDD-NOS | 48 |
| AUTISM | 41 |
| ASPERGER'S DISORDER | 15 |

Since some diagnostic subgroups contained a limited number of samples, especially **Asperger's Disorder**, performing a multi-class classification task could decrease statistical power and increase the possibility of model overfitting. Therefore, all ASD-related diagnoses (**AUTISM**, **PDD-NOS**, and **ASPERGER'S DISORDER**) were combined into one unified ASD class, while **CONTROL** samples were assigned to the healthy control class.

This conversion resulted in a binary classification problem:

- **Control (Label = 0):** Healthy individuals  
- **ASD (Label = 1):** Individuals with autism spectrum-related diagnoses  

This strategy enables a more robust classification model by increasing the number of samples in the disease group and focusing on the primary objective of ASD detection rather than distinguishing between different ASD subtypes.

In [12]:
def convert_label(diagnosis):
    if "CONTROL" in diagnosis:
        return 0
    else:
        return 1


df_gpl6244["Label"] = df_gpl6244["Diagnosis"].apply(convert_label)

df_gpl6244.head()

,Sample,Diagnosis,Gender,Age,Platform,Label
99,GSM650655,ASPERGER'S DISORDER,male,156,GPL6244,1
100,GSM650656,AUTISM,male,168,GPL6244,1
101,GSM650657,CONTROL,male,228,GPL6244,0
102,GSM650658,AUTISM,male,60,GPL6244,1
103,GSM650659,AUTISM,male,36,GPL6244,1


In [13]:
df_gpl6244["Label"].value_counts()

Label
1    104
0     82
Name: count, dtype: int64

In [14]:
df_gpl6244["Diagnosis"].value_counts()

Diagnosis
CONTROL                82
PDD-NOS                48
AUTISM                 41
ASPERGER'S DISORDER    15
Name: count, dtype: int64

### **Gender Distribution Analysis**

The gender distribution of the selected GPL6244 dataset was examined to evaluate the demographic composition of the samples. The dataset contains **128 male samples** and **58 female samples**.

The distribution across diagnostic groups shows that the control group (**Label = 0**) includes **48 males** and **34 females**, while the ASD group (**Label = 1**) contains **80 males** and **24 females**. This analysis highlights the gender imbalance commonly observed in ASD datasets, where males are more frequently represented than females.

In [15]:
print(df_gpl6244['Gender'].value_counts())

Gender
male      128
female     58
Name: count, dtype: int64


In [16]:
print(df_gpl6244.groupby(['Label', 'Gender']).size())

Label  Gender
0      female    34
       male      48
1      female    24
       male      80
dtype: int64


### **Initial Statistical Description**

The dataset contains 186 samples with complete metadata and no missing values.  
The samples were obtained from the GPL6244 platform. The dataset includes 128 male and 58 female participants.  

The age of participants ranged from 24 to 264 months, with a mean age of 96.8 months.  
The median age was 84 months, indicating a wide age distribution among samples.

All samples were generated using the same microarray platform, ensuring consistency across the dataset.

In [17]:
df_gpl6244.isnull().sum()

Sample       0
Diagnosis    0
Gender       0
Age          0
Platform     0
Label        0
dtype: int64

In [18]:
df_gpl6244.describe(include="all")

,Sample,Diagnosis,Gender,Age,Platform,Label
count,186,186,186,186.000000,186,186.00000
unique,186,4,2,NaN,1,NaN
top,GSM650655,CONTROL,male,NaN,GPL6244,NaN
freq,1,82,128,NaN,186,NaN
mean,NaN,NaN,NaN,96.806452,NaN,0.55914
std,NaN,NaN,NaN,52.409460,NaN,0.49783
min,NaN,NaN,NaN,24.000000,NaN,0.00000
25%,NaN,NaN,NaN,51.250000,NaN,0.00000
50%,NaN,NaN,NaN,84.000000,NaN,1.00000
75%,NaN,NaN,NaN,136.500000,NaN,1.00000


### **Data Exploration**

- **Load GEO Dataset** 
- **Inspect Dataset Structure** 
- **Extract Sample Metadata** 
- **Check Platforms** 
- **Select GPL6244 Platform** 
- **Convert Diagnosis to Binary Labels** 
- **Analyze Diagnosis Distribution** 
- **Analyze Gender Distribution** 
- **Missing Value Analysis** 
- **Initial Statistical Description** 


### **Load Platform Annotation**

The platform annotation (GPL6244) was loaded to retrieve the probe annotation table. This table provides the mapping between probe identifiers and their corresponding gene annotations, which is required to convert probe-level expression data into gene-level expression data.


In [19]:
platform = gse.gpls['GPL6244']
type(platform)

platform_table = platform.table

platform_table.head()

,ID,GB_LIST,SPOT_ID,seqname,RANGE_GB,RANGE_STRAND,RANGE_START,RANGE_STOP,total_probes,gene_assignment,mrna_assignment,category
0,7896736,NaN,chr1:53049-54936,chr1,NC_000001.10,+,53049,54936,7,---,NONHSAT060105 // NONCODE // Non-coding transcr...,main
1,7896738,NaN,chr1:63015-63887,chr1,NC_000001.10,+,63015,63887,31,ENST00000328113 // OR4G2P // olfactory recepto...,ENST00000328113 // ENSEMBL // havana:known chr...,main
2,7896740,"NM_001004195,NM_001005240,NM_001005484,BC13684...",chr1:69091-70008,chr1,NC_000001.10,+,69091,70008,24,"NM_001004195 // OR4F4 // olfactory receptor, f...",NM_001004195 // RefSeq // Homo sapiens olfacto...,main
3,7896742,"NR_024437,XM_006711854,XM_006726377,XR_430662,...",chr1:334129-334296,chr1,NC_000001.10,+,334129,334296,6,NR_024437 // LOC728323 // uncharacterized LOC7...,NR_024437 // RefSeq // Homo sapiens uncharacte...,main
4,7896744,"NM_001005221,NM_001005224,NM_001005277,NM_0010...",chr1:367659-368597,chr1,NC_000001.10,+,367659,368597,36,"NM_001005221 // OR4F29 // olfactory receptor, ...",NM_001005221 // RefSeq // Homo sapiens olfacto...,main


In [20]:
platform.table.columns.tolist()
annotation = platform.table[['ID', 'gene_assignment']]
annotation.head()

,ID,gene_assignment
0,7896736,---
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f..."
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ..."


### **Load Gene Expression Data**

The expression data for a single sample (`GSM650655`) was loaded. Each row represents the expression value measured for a specific probe on the microarray.

In [21]:
sample = gse.gsms['GSM650655'].table

sample.head()

,ID_REF,VALUE
0,7892501,9.67160
1,7892502,122.91175
2,7892503,151.17249
3,7892504,1935.89058
4,7892505,21.14810


### **Map Probes to Gene Annotations**

The expression data was merged with the platform annotation using the probe identifiers. This step associates each probe with its corresponding gene annotation.


In [22]:
merged = sample.merge(
    annotation,
    left_on='ID_REF',
    right_on='ID',
    how='left'
)
merged = merged.drop(columns=['ID'])
merged.head()

,ID_REF,VALUE,gene_assignment
0,7892501,9.67160,---
1,7892502,122.91175,---
2,7892503,151.17249,---
3,7892504,1935.89058,---
4,7892505,21.14810,---


In [23]:
merged = merged[['gene_assignment','VALUE']]

merged.head()

,gene_assignment,VALUE
0,---,9.67160
1,---,122.91175
2,---,151.17249
3,---,1935.89058
4,---,21.14810


In [24]:
gene_expression = merged.groupby('gene_assignment')['VALUE'].mean()
gene_expression = gene_expression[gene_expression.index != '---']

In [25]:
print(annotation.columns)
print(df_gpl6244.columns)

Index(['ID', 'gene_assignment'], dtype='str')
Index(['Sample', 'Diagnosis', 'Gender', 'Age', 'Platform', 'Label'], dtype='str')


### **Extract Gene Symbols from Annotation**

The `gene_assignment` column contains messy strings with multiple fields separated by `//`. This step extracts only the gene symbol (the second field).

For example, a string like `"NM_001123456 // TP53 // protein binding"` is split into:
- `parts[0]` → `"NM_001123456 "` (accession number)
- `parts[1]` → `" TP53 "` (gene symbol) → after `.strip()` → `"TP53"`
- `parts[2]` → `" protein binding"` (description)

In [26]:
def extract_gene(x):
    if pd.isna(x):
        return None
    
    parts = x.split('//')
    
    if len(parts) > 1:
        return parts[1].strip()
    
    return None


annotation['gene_symbol'] = annotation['gene_assignment'].apply(extract_gene)

annotation.head()

,ID,gene_assignment,gene_symbol
0,7896736,---,NaN
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...,OR4G2P
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f...",OR4F4
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...,LOC728323
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ...",OR4F29


In [27]:
annotation['gene_symbol'] = annotation['gene_assignment'].apply(
    lambda x: x.split('//')[1].strip() if isinstance(x, str) and '//' in x else None
)

print(annotation['gene_symbol'].isna().sum())
print(annotation['gene_symbol'])
print(annotation['gene_symbol'].nunique())
print(annotation.shape)

8004
0              NaN
1           OR4G2P
2            OR4F4
3        LOC728323
4           OR4F29
           ...    
33292          NaN
33293          NaN
33294          NaN
33295          NaN
33296          NaN
Name: gene_symbol, Length: 33297, dtype: str
23307
(33297, 3)


In [28]:
annotation.head()

,ID,gene_assignment,gene_symbol
0,7896736,---,NaN
1,7896738,ENST00000328113 // OR4G2P // olfactory recepto...,OR4G2P
2,7896740,"NM_001004195 // OR4F4 // olfactory receptor, f...",OR4F4
3,7896742,NR_024437 // LOC728323 // uncharacterized LOC7...,LOC728323
4,7896744,"NM_001005221 // OR4F29 // olfactory receptor, ...",OR4F29


### **Build Gene Expression Matrix for GPL6244 Samples**

After extracting gene symbols from the platform annotation, the labels are prepared by extracting the `Sample` and `Label` columns from `df_gpl6244` and setting the sample ID as the index. Then, only samples belonging to the **GPL6244** platform are filtered into a dictionary.

For each of these 186 samples, the probe-level expression table is merged with the annotation table on probe IDs. Rows without a valid `gene_symbol` are dropped, and probes mapping to the same gene are aggregated by **mean expression value**. The resulting gene-level expression Series for each sample is stored in a list.

Finally, all Series are concatenated column-wise and transposed to create a matrix where **rows = samples (186)** and **columns = genes (23,307)**, ready for machine learning.

In [29]:
labels = df_gpl6244[['Sample','Label']]
labels = labels.set_index('Sample')
labels.head()

,Label
Sample,
GSM650655,1
GSM650656,1
GSM650657,0
GSM650658,1
GSM650659,1


In [30]:
samples_gpl6244 = {}

for gsm_id, gsm in gse.gsms.items():
    if gsm.metadata['platform_id'][0] == 'GPL6244':
        samples_gpl6244[gsm_id] = gsm


expression_list = []

for gsm_id, gsm in samples_gpl6244.items():

    sample_table = gsm.table

    merged = sample_table.merge(
        annotation,
        left_on='ID_REF',
        right_on='ID',
        how='left'
    )

    merged = merged.dropna(subset=['gene_symbol'])

    gene_exp = (
        merged
        .groupby('gene_symbol')['VALUE']
        .mean()
    )

    gene_exp.name = gsm_id

    expression_list.append(gene_exp)

In [31]:
expression_matrix = pd.concat(expression_list, axis=1)
X = expression_matrix.T
X.shape

(186, 23307)

In [32]:
print(X.columns[:10].tolist())
print(len(X.columns))

['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC']
23307


### **Preprocessing Steps Completed**

So far, the following preprocessing steps have been performed:

- **Loaded raw expression data** from GSE18123 using GEOparse
- **Extracted metadata** (diagnosis, gender, age, platform) for all 285 samples
- **Selected GPL6244 platform** (186 samples) over GPL570 (99 samples)
- **Converted multi-class diagnoses** (CONTROL, AUTISM, PDD-NOS, ASPERGER) into **binary labels** (Control = 0, ASD = 1)
- **Loaded platform annotation** (GPL6244) containing probe-to-gene mappings
- **Extracted gene symbols** from the messy `gene_assignment` column using string splitting
- **Built gene expression matrix** by merging each sample's probe-level data with the annotation, aggregating probes to genes by mean, and concatenating all 186 samples into a matrix of shape **(186 samples × 23,307 genes)**

### **Remove Sex Chromosome Genes**

Sex chromosome genes (Y-linked and X-linked) are removed from the expression matrix to prevent gender-related bias from influencing the classification model. Since the dataset has an imbalanced gender distribution (128 males vs. 58 females), keeping these genes could lead the model to learn gender-specific patterns rather than autism-related biomarkers.


In [48]:
sex_genes = [
    'DDX3Y', 'USP9Y', 'UTY', 'TMSB4Y', 'KDM5D', 
    'EIF1AY', 'RPS4Y1', 'RPS4Y2', 'ZFY', 'SRY',
    'PRKY', 'BPY2', 'DAZ1', 'DAZ2', 'DAZ3', 'DAZ4',
    'CDY1', 'CDY2', 'HSFY1', 'HSFY2',
    'XIST', 'TSIX', 'JPX', 'FTX'
]

In [49]:
cols_to_drop = [col for col in sex_genes if col in X.columns]

print(f"Sex genes found:{len(cols_to_drop)}")
print(f"List: {cols_to_drop}")

Sex genes found:14
List: ['DDX3Y', 'USP9Y', 'UTY', 'TMSB4Y', 'KDM5D', 'EIF1AY', 'RPS4Y1', 'RPS4Y2', 'ZFY', 'SRY', 'PRKY', 'DAZ4', 'HSFY1', 'JPX']


**Result:** 14 sex chromosome genes were found and removed, reducing the feature set from **23,307** to **23,293** genes.

### **Join Labels with Expression Data**

The filtered expression matrix is joined with the binary labels to create the final dataset for modeling.


In [50]:
df_filtered = X.drop(columns=cols_to_drop)

print(f"Genes before: {X.shape[1]}")
print(f"Genes after: {df_filtered.shape[1]}")

Genes before: 23307
Genes after: 23293


In [36]:
data = df_filtered.join(labels)

In [40]:
print(data.shape)
print(data.columns[:10].tolist())
print(data['Label'].value_counts())

(186, 23294)
['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC']
Label
1    104
0     82
Name: count, dtype: int64


### **Feature Selection using Statistical Testing**

To reduce dimensionality and identify the most relevant genes for ASD classification, a two-step feature selection approach is applied.

#### **Step 1: Differential Expression Analysis (t-test)**

First, a quick t-test is performed on a few genes to check for expression differences between Control and ASD groups.



In [56]:
label_col = 'Label'
genes = [col for col in data.columns if col != label_col]

print(f"Total Genes: {len(genes)}")

p_values = []

for i, gene in enumerate(genes[:5]):  
    group0 = data[data[label_col] == 0][gene]
    group1 = data[data[label_col] == 1][gene]
    _, p = ttest_ind(group0, group1)
    p_values.append(p)
    print(f" Gene {gene}: p-value = {p:.6f}")

Total Genes: 23293
 Gene A1BG: p-value = 0.023582
 Gene A1CF: p-value = 0.335687
 Gene A2M: p-value = 0.353292
 Gene A2ML1: p-value = 0.105601
 Gene A3GALT2: p-value = 0.708521



**Sample results:** Gene A1BG shows significant differential expression (p = 0.024), while others like A1CF and A2M are not significant.

#### **Step 2: ANOVA F-test with FDR Correction**

Instead of testing genes one by one, `f_classif` is used to compute F-statistics and p-values for all 23,293 genes simultaneously. To control for multiple hypothesis testing, the **Benjamini-Hochberg FDR** method is applied (alpha = 0.05).

In [57]:
label_col = 'Label'
genes = [col for col in data.columns if col != label_col]

f_stats, p_values = f_classif(data[genes], data[label_col])

reject, p_adjusted, _, _ = multipletests(p_values, method='fdr_bh', alpha=0.05)

significant_genes = [genes[i] for i in range(len(genes)) if reject[i]]

print(f"Total genes: {len(genes)}")
print(f"Significant genes: {len(significant_genes)}")

Total genes: 23293
Significant genes: 6611



**Result:** Out of 23,293 genes, **6,611 genes** passed the FDR correction and are considered significantly differentially expressed between Control and ASD groups.

#### **Step 3: Variance Filtering**

From the 6,611 significant genes, the **top 500 most variable genes** are selected based on expression variance across all samples. This removes low-variance genes that may have limited discriminative power.

In [55]:
sig_data = data[significant_genes]

variances = sig_data.var()

top_var_genes = variances.nlargest(500).index.tolist()

print(f"Genes after variance filter:{len(top_var_genes)}")

Genes after variance filter:500



**Final feature set:** **500 genes** selected for downstream classification modeling.

### **Lasso-based Feature Selection**

After reducing the feature set to 500 highly variable genes, **Lasso (L1 Regularization)** is applied to further shrink the feature space. Lasso penalizes the absolute size of coefficients, driving irrelevant features to exactly zero, leaving only the most informative genes for classification.



In [59]:
X_selected = data[top_var_genes]
y = data['Label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

lasso = LassoCV(cv=5, random_state=42, max_iter=10000, tol=1e-3) 
lasso.fit(X_scaled, y)

selected_indices = np.where(lasso.coef_ != 0)[0]
final_genes = [top_var_genes[i] for i in selected_indices]

print(f"Final number of genes: {len(final_genes)}")
print(f"Final genes: {final_genes}")

Final number of genes: 20
Final genes: ['SCARNA17', 'BCL2L1', 'RPS26P15', 'MALAT1', 'MS4A1', 'SNORD95', 'FTH1P11', 'SNORD3B-1', 'MARCH8', 'FCN1', 'AOAH', 'IGKV3D-11', 'SNORD50A', 'HLA-DQA2', 'HSP90AB1', 'DUSP6', 'CD36', 'ZNF836', 'KRTAP19-5', 'PCMTD1']



**Result:** Lasso with 5-fold cross-validation selected **20 genes** with non-zero coefficients from the initial 500 candidates, providing a compact and interpretable set of biomarkers for ASD classification.
